**Fig. 2, step 1: sample axial (XZ + YZ) slices from the 7 development stacks and train one axial model per HITL round, rounds 0 to 9**

Needs a GPU.

In [ ]:
# Working directory: data/fig2/ under the repository root, where the Fig. 2 deposit folder is downloaded
# (training_data_spatial/, training_data_axial/, training_rounds_results/, cross_round_metrics/).
# Set the EPICYCLE_ANALYSES_DATA environment variable to keep the deposit somewhere else.
import os
from pathlib import Path
_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "environment.yml").exists() and (p / "data").is_dir()), None)
assert _root is not None, "start Jupyter from inside the EpiCycle-analyses repository"
os.chdir(Path(os.environ.get("EPICYCLE_ANALYSES_DATA", _root / "data")) / "fig2")
print("working directory:", os.getcwd())


In [ ]:
import numpy as np
import tifffile
from skimage.segmentation import find_boundaries
from skimage.measure import label, regionprops
from skimage.morphology import opening, closing, disk
from typing import Tuple, List


In [ ]:
class BootstrappedSliceLoader:
    """One ground-truth stack (image + instance mask) exposed as 2D slices of one view.

    The stack is loaded once and transposed so that axis 0 indexes the slices of the
    requested view: XY (lateral), XZ or YZ (axial). `get_valid_indices` gives the slice
    indices that may be sampled; `get_slice` returns one image (C, H, W) + mask (H, W) pair.
    Mask clean-up (`clean_mask_stack`) is applied to the whole stack only when
    `relabel_masks=True`; the round sampler keeps it off and cleans the sampled slices
    one by one instead.
    """
    def __init__(
        self,
        image_path: str,
        mask_path: str,
        view: str = "XY",
        relabel_masks: bool = True,
        min_size: int = 100,
        smoothing_disk_size: int = 1,
        edge_buffer: int = 20
    ):
        self.image_path = image_path
        self.mask_path = mask_path
        self.view = view.upper()
        self.relabel_masks = relabel_masks
        self.min_size = min_size
        self.smoothing_disk_size = smoothing_disk_size
        self.edge_buffer = edge_buffer

        # Load and process
        self.images, self.masks = self.load_and_prepare()
        self.depth = self.masks.shape[0]  # Number of slices

    def load_and_prepare(self) -> Tuple[np.ndarray, np.ndarray]:
        """Read the TIFFs and transpose them to (slices, C, H, W) / (slices, H, W) for `self.view`.

        Accepts images stored as (Z, C, H, W) or (C, Z, H, W); the image must have exactly
        two channels (membrane, nucleus).
        """
        image = tifffile.imread(self.image_path)  # (Z, C, H, W) or (C, Z, H, W)
        mask = tifffile.imread(self.mask_path)    # (Z, H, W)

        # Fix channel order if needed
        if image.shape[0] == 2 and image.shape[1] != 2:
            image = np.transpose(image, (1, 0, 2, 3))  # (Z, C, H, W)
        if image.shape[1] != 2:
            raise ValueError("Expected image with 2 channels.")

        # Transpose for the desired view
        if self.view == "XY":
            pass  # (Z, C, H, W), (Z, H, W)
        elif self.view == "XZ":
            mask = np.transpose(mask, (1, 0, 2))        # (X, Z, Y)
            image = np.transpose(image, (1, 2, 0, 3))   # (C, X, Z, Y)
            mask = np.transpose(mask, (0, 1, 2))        # (D, H, W)
            image = np.transpose(image, (1, 0, 2, 3))   # (D, C, H, W)
        elif self.view == "YZ":
            mask = np.transpose(mask, (2, 0, 1))        # (Y, Z, X)
            image = np.transpose(image, (1, 3, 0, 2))   # (C, Y, Z, X)
            mask = np.transpose(mask, (0, 1, 2))        # (D, H, W)
            image = np.transpose(image, (1, 0, 2, 3))   # (D, C, H, W)
        else:
            raise ValueError(f"Invalid view: {self.view}")

        if self.relabel_masks:
            mask = self.clean_mask_stack(mask)

        return image, mask

    def clean_mask_stack(self, mask_stack: np.ndarray) -> np.ndarray:
        """Clean every slice of an instance mask stack (Methods §3.3).

        Per slice: zero the inner object boundaries so touching labels separate, relabel
        connected components (4-connectivity), drop objects smaller than `min_size` pixels,
        and smooth each kept object by a morphological opening then closing with a disk of
        radius `smoothing_disk_size`.
        """
        cleaned_stack = np.zeros_like(mask_stack, dtype=np.uint16)

        for i in range(mask_stack.shape[0]):
            slice_ = mask_stack[i]

            # Remove boundaries
            boundary = find_boundaries(slice_, mode='inner')
            cleaned = slice_.copy()
            cleaned[boundary] = 0

            # Relabel connected components
            relabeled = label(cleaned, connectivity=1)
            smoothed = np.zeros_like(relabeled)

            selem = disk(self.smoothing_disk_size) if self.smoothing_disk_size > 0 else None

            for region in regionprops(relabeled):
                if region.area >= self.min_size:
                    mask = (relabeled == region.label)
                    if selem is not None:
                        mask = opening(mask, selem)
                        mask = closing(mask, selem)
                    smoothed[mask] = region.label

            cleaned_stack[i] = smoothed

        return cleaned_stack

    def get_valid_indices(self, total: int = 100) -> List[int]:
        """Return `total` evenly spaced candidate slice indices.

        Axial views (XZ, YZ) exclude `edge_buffer` slices at both ends of the stack, where
        there is little or no tissue; the lateral view (XY) uses the full depth.
        """
        if self.view in ["XZ", "YZ"]:
            valid = np.arange(self.edge_buffer, self.depth - self.edge_buffer)
        else:
            valid = np.arange(0, self.depth)

        return np.linspace(valid[0], valid[-1], total, dtype=int).tolist()

    def get_slice(self, index: int) -> Tuple[np.ndarray, np.ndarray]:
        """Return the image (C, H, W) and raw mask (H, W) of one slice.
        """
        img = self.images[index]   # (C, H, W)
        mask = self.masks[index]   # (H, W)
        return img, mask


In [ ]:
import os
import json
import shutil
import numpy as np
import tifffile
from typing import List, Dict
from tqdm.notebook import tqdm
from skimage.segmentation import find_boundaries
from skimage.measure import label, regionprops
from skimage.morphology import opening, closing, disk


def sample_slices_for_round(
    round_number: int,
    stacks: List[Dict],         # List of dicts: {name, image_path, mask_path}
    views: List[str],           # e.g. ["XY", "XZ", "YZ"]
    slices_per_stack: int = 10,
    min_distance: int = 5,
    edge_buffer: int = 20,
    allowed_per_view: int = 100,
    base_dir: str = "training_data"
):
    """Sample this round's new training slices from every stack and write the cumulative round folder.

    For each view and stack: take the candidate indices from `BootstrappedSliceLoader`,
    remove those drawn in earlier rounds (read from the previous round's
    `selected_indices.json`), and draw `slices_per_stack` of the rest in random order,
    keeping at least `min_distance` slices between the picks of this round. Each picked
    slice is cleaned (boundary removal, relabel, min size, open/close; Methods §3.3) and
    written as `<VIEW>_<stack>_<index>_img.tif` (float32) + `_mask.tif` (uint16) into
    `base_dir/round_N/spatial/` (XY) or `.../axial/` (XZ, YZ). The slices of all earlier
    rounds are copied in as well, so `round_N` holds the cumulative training set.

    Returns (all indices selected so far, indices new this round), both as
    {VIEW: {stack: [indices]}}; the former is also saved as `round_N/selected_indices.json`.
    Sampling is unseeded.
    """
    round_dir = os.path.join(base_dir, f"round_{round_number}")
    spatial_dir = os.path.join(round_dir, "spatial")
    axial_dir = os.path.join(round_dir, "axial")
    os.makedirs(spatial_dir, exist_ok=True)
    os.makedirs(axial_dir, exist_ok=True)

    # Load previous selected indices
    prev_indices_path = os.path.join(base_dir, f"round_{round_number - 1}", "selected_indices.json")
    if os.path.exists(prev_indices_path):
        with open(prev_indices_path, "r") as f:
            selected_indices = json.load(f)
    else:
        selected_indices = {view.upper(): {} for view in views}

    new_indices = {view.upper(): {} for view in views}

    for view in views:
        view_upper = view.upper()
        for stack in tqdm(stacks, desc=f"Sampling {view_upper}"):
            patch = stack["name"]

            loader = BootstrappedSliceLoader(
                image_path=stack["image_path"],
                mask_path=stack["mask_path"],
                view=view_upper,
                relabel_masks=False,  # <-- Don't clean full stack!
                min_size=100,
                smoothing_disk_size=1,
                edge_buffer=edge_buffer
            )

            # Get valid (non-edge) indices
            allowed = loader.get_valid_indices(total=allowed_per_view)
            used = selected_indices.get(view_upper, {}).get(patch, []).copy()

            available = [i for i in allowed if i not in used]
            selected = []
            for idx in np.random.permutation(available):
                if all(abs(idx - s) >= min_distance for s in selected):
                    selected.append(idx)
                    if len(selected) == slices_per_stack:
                        break

            # Track selections
            new_indices[view_upper].setdefault(patch, []).extend(selected)
            selected_indices.setdefault(view_upper, {}).setdefault(patch, []).extend(selected)

            # Save selected slices (cleaned)
            for idx in selected:
                img, raw_mask = loader.get_slice(idx)

                # === Clean only this slice ===
                boundary = find_boundaries(raw_mask, mode='inner')
                cleaned = raw_mask.copy()
                cleaned[boundary] = 0
                relabeled = label(cleaned, connectivity=1)
                final_mask = np.zeros_like(relabeled)

                selem = disk(loader.smoothing_disk_size) if loader.smoothing_disk_size > 0 else None

                for region in regionprops(relabeled):
                    if region.area >= loader.min_size:
                        region_mask = (relabeled == region.label)
                        if selem is not None:
                            region_mask = opening(region_mask, selem)
                            region_mask = closing(region_mask, selem)
                        final_mask[region_mask] = region.label

                # === Save slice ===
                fname_img = f"{view_upper}_{patch}_{idx:03d}_img.tif"
                fname_mask = f"{view_upper}_{patch}_{idx:03d}_mask.tif"
                save_dir = spatial_dir if view_upper == "XY" else axial_dir

                tifffile.imwrite(os.path.join(save_dir, fname_img), img.astype(np.float32))
                tifffile.imwrite(os.path.join(save_dir, fname_mask), final_mask.astype(np.uint16))

            # === Copy previous round slices ===
            if round_number > 0:
                prev_round_dir = os.path.join(base_dir, f"round_{round_number - 1}")
                prev_view_dir = os.path.join(prev_round_dir, "spatial" if view_upper == "XY" else "axial")
                for prev_idx in used:
                    for suffix in ["img", "mask"]:
                        fname = f"{view_upper}_{patch}_{prev_idx:03d}_{suffix}.tif"
                        src = os.path.join(prev_view_dir, fname)
                        dst = os.path.join(spatial_dir if view_upper == "XY" else axial_dir, fname)
                        if os.path.exists(src):
                            shutil.copy2(src, dst)

    # === Save updated index log with numpy-safe JSON ===
    def convert_numpy(obj):
        if isinstance(obj, (np.integer, np.int64)): return int(obj)
        if isinstance(obj, (np.floating, np.float32, np.float64)): return float(obj)
        if isinstance(obj, np.ndarray): return obj.tolist()
        raise TypeError(f"Object of type {type(obj)} is not JSON serializable")

    with open(os.path.join(round_dir, "selected_indices.json"), "w") as f:
        json.dump(selected_indices, f, indent=4, default=convert_numpy)

    return selected_indices, new_indices


In [ ]:
# Point to your /data directory
data_dir = "../ground_truth/zebrafish/development"   # the 7 development stacks: patch-N.tif + patch-N_masks.tif (shared with Fig. 3)

# Automatically detect patch pairs
patch_names = sorted(set(
    f.replace("_masks.tif", "").replace(".tif", "")
    for f in os.listdir(data_dir) if f.endswith(".tif")
))

# Filter out mask-only files
patch_names = [p for p in patch_names if not p.endswith("_masks")]

# Build stack list
stacks = []
for patch in patch_names:
    stacks.append({
        "name": patch,
        "image_path": os.path.join(data_dir, f"{patch}.tif"),
        "mask_path": os.path.join(data_dir, f"{patch}_masks.tif")
    })

# Confirm it worked
for s in stacks:
    print(f"{s['name']}:")
    print(f"  image: {s['image_path']}")
    print(f"  mask : {s['mask_path']}")


patch-1:
  image: ./data/patch-1.tif
  mask : ./data/patch-1_masks.tif
patch-2:
  image: ./data/patch-2.tif
  mask : ./data/patch-2_masks.tif
patch-3:
  image: ./data/patch-3.tif
  mask : ./data/patch-3_masks.tif
patch-4:
  image: ./data/patch-4.tif
  mask : ./data/patch-4_masks.tif
patch-5:
  image: ./data/patch-5.tif
  mask : ./data/patch-5_masks.tif
patch-7:
  image: ./data/patch-7.tif
  mask : ./data/patch-7_masks.tif
patch-8:
  image: ./data/patch-8.tif
  mask : ./data/patch-8_masks.tif


In [ ]:
import os
import tifffile
import numpy as np

def load_training_pairs(round_number, base_dir="training_data", mode="both"):
    """Load every image/mask pair of `base_dir/round_N/<spatial|axial>` as two lists.

    Files are paired by sorted name (`*_img.tif` with `*_mask.tif`). `mode` selects the
    lateral folder, the axial folder, or both.
    """
    assert mode in ["spatial", "axial", "both"], "Invalid mode. Choose 'spatial', 'axial', or 'both'."

    round_dir = os.path.join(base_dir, f"round_{round_number}")
    dirs_to_load = []

    if mode in ["spatial", "both"]:
        dirs_to_load.append(os.path.join(round_dir, "spatial"))
    if mode in ["axial", "both"]:
        dirs_to_load.append(os.path.join(round_dir, "axial"))

    all_images = []
    all_masks = []

    for subdir in dirs_to_load:
        img_files = sorted([f for f in os.listdir(subdir) if f.endswith("_img.tif")])
        mask_files = sorted([f for f in os.listdir(subdir) if f.endswith("_mask.tif")])

        assert len(img_files) == len(mask_files), f"Mismatch in image/mask counts in {subdir}"

        for img_f, mask_f in zip(img_files, mask_files):
            img = tifffile.imread(os.path.join(subdir, img_f))     # (C, H, W)
            mask = tifffile.imread(os.path.join(subdir, mask_f))   # (H, W)

            all_images.append(img)
            all_masks.append(mask)

    print(f"Loaded {len(all_images)} image/mask pairs from Round {round_number} ({mode})")
    return all_images, all_masks



In [ ]:
import os
import time
from datetime import timedelta
import matplotlib.pyplot as plt
import numpy as np
from cellpose import models, train


def train_cellpose_model(
        train_data,
        train_labels,
        channels,
        model_save_path,
        n_epochs,
        learning_rate,
        weight_decay,
        model_name,
        batch_size,
        normalize,
        min_train_masks,
        momentum,
        rescale=True,
        SGD=True,
        compute_flows=True,
        use_GPU=True,
        return_model=False,
        val_data=None,
        val_labels=None,
        pretrained_model_path=None # NEW PARAM
):
    """Train (or fine-tune) one Cellpose model on in-memory slices and save it under
    `model_save_path/models/model_name`.

    If that checkpoint and its loss curve exist the model is loaded instead of retrained (this is what
    makes the round loop resumable). `pretrained_model_path` selects fine-tuning from an
    existing checkpoint; otherwise the network starts from scratch (`model_type=None`).
    Training uses `cellpose.train.train_seg` with the given SGD settings. The training loss
    curve is saved as `<model_name>_train_loss.npy`; a validation curve is saved only when
    `val_data` / `val_labels` are passed (the HITL rounds pass none).
    """
    import os
    import numpy as np
    from cellpose import models, train

    # Ensure output directory exists
    if not os.path.exists(model_save_path):
        print('Model directory does not exist, creating directory...')
        os.makedirs(model_save_path)

    model_save_dir = os.path.join(model_save_path, 'models', model_name)
    if os.path.exists(model_save_dir) and os.path.exists(model_save_dir + '_train_loss.npy'):  # no loss curve = a killed run: retrain
        print(f'Model already exists. Loading instead of training: {model_name}')
        model = models.CellposeModel(
            gpu=use_GPU,
            pretrained_model=model_save_dir
        )
        return model if return_model else None

    # === Load previous model (for fine-tuning) OR train from scratch
    if pretrained_model_path:
        print(f"Fine-tuning from previous model: {pretrained_model_path}")
        model = models.CellposeModel(gpu=use_GPU, pretrained_model=pretrained_model_path)
    else:
        print("Training model from scratch.")
        model = models.CellposeModel(gpu=use_GPU, model_type=None)

    print(f"Training model: {model_name}")

    # === Train the model
    new_model_path, train_loss, val_loss = train.train_seg(
        net=model.net,
        train_data=train_data,
        train_labels=train_labels,
        channels=channels,
        save_path=model_save_path,
        n_epochs=n_epochs,
        learning_rate=learning_rate,
        weight_decay=weight_decay,
        batch_size=batch_size,
        normalize=normalize,
        min_train_masks=min_train_masks,
        model_name=model_name,
        momentum=momentum,
        rescale=rescale,
        SGD=SGD,
        compute_flows=compute_flows,
        test_data=val_data if val_data is not None else None,
        test_labels=val_labels if val_labels is not None else None
    )

    # Save training and validation loss curves
    np.save(os.path.join(model_save_path, "models", model_name + '_train_loss.npy'), train_loss)
    if val_data or val_labels:
        np.save(os.path.join(model_save_path, "models", model_name + '_val_loss.npy'), val_loss)

    print(f"Trained Model saved to: {model_save_path}")
    return model if return_model else None


In [ ]:
import shutil
import time
from datetime import timedelta
import os

# ========== CONFIGURATION ==========
num_rounds = 10                    # Total number of rounds
slices_per_stack = 15              # New slices per stack per round
edge_buffer = 50
min_distance = 5
allowed_per_view = 100             # Max candidates per stack per view
channels = [2, 1]                  # Membrane, nuclei
weight_decay = 1e-5
batch_size = 32
momentum = 0.9
min_train_masks = 0
use_dynamic_lr = False             # fixed LR 0.01 every round, as run

# === Choose training mode ===
train_mode = "axial"   # or "spatial"

# === Automatically set paths based on mode ===
if train_mode.lower() == "spatial":
    model_save_path = 'training_rounds_results/spatial_models/'
    training_data_dir = 'training_data_spatial'
elif train_mode.lower() == "axial":
    model_save_path = 'training_rounds_results/axial_models/'
    training_data_dir = 'training_data_axial'
else:
    raise ValueError(f"Unknown train_mode: {train_mode}")

print(f"Model save path: {model_save_path}")
print(f"Training data path: {training_data_dir}")


# ========= Training Schedule Function =========
def get_training_schedule(round_number: int):
    """Epochs and learning rate for one round: 1,000 epochs at a fixed LR of 0.01 (as run)."""
    n_epochs = 1000

    if use_dynamic_lr:
        learning_rate = 1e-2 if round_number < 5 else 1e-3
    else:
        learning_rate = 1e-2

    return n_epochs, learning_rate


# ========= Determine Resume Point =========
completed_rounds = 0
for r in range(num_rounds):
    model_dir = os.path.join(model_save_path, 'models', f"{train_mode}_model_round{r}")
    if os.path.exists(model_dir) and os.path.exists(model_dir + '_train_loss.npy'):  # no loss curve = a killed run: redo the round
        completed_rounds += 1
    else:
        break  # Stop at the first incomplete round

print(f"Resuming {train_mode} training from Round {completed_rounds}...")


# ========= Training Loop =========
for round_number in range(completed_rounds, num_rounds):
    print(f"\n========== ROUND {round_number} ({train_mode.upper()}) ==========")

    # Clean up any partial previous round folder
    round_dir = os.path.join(training_data_dir, f"round_{round_number}")
    if os.path.exists(round_dir):
        print(f"Incomplete data detected for Round {round_number}. Removing old data...")
        shutil.rmtree(round_dir)

    # ---- 1. Set view mode ----
    if train_mode.lower() == "spatial":
        views = ["XY"]
    else:
        views = ["XZ", "YZ"]

    # ---- 2. Sample slices ----
    selected_indices, new_indices = sample_slices_for_round(
        round_number=round_number,
        stacks=stacks,
        views=views,
        slices_per_stack=slices_per_stack,
        min_distance=min_distance,
        edge_buffer=edge_buffer,
        allowed_per_view=allowed_per_view,
        base_dir=training_data_dir
    )

    # ---- 3. Load data ----
    imgs, masks = load_training_pairs(round_number, base_dir=training_data_dir, mode=train_mode)

    # ---- 4. Training schedule ----
    n_epochs, learning_rate = get_training_schedule(round_number)
    print(f"Training schedule for round {round_number}: {n_epochs} epochs @ LR={learning_rate}")

    # ---- 5. Determine pretrained model (for fine-tuning) ----
    if round_number > 0:
        prev_model_name = f"{train_mode}_model_round{round_number - 1}"
        prev_model_path = os.path.join(model_save_path, "models", prev_model_name)
        if not os.path.exists(prev_model_path):
            print(f"Previous model not found: {prev_model_path}. Training from scratch.")
            prev_model_path = None
        else:
            print(f"Fine-tuning from: {prev_model_path}")
    else:
        prev_model_path = None  # Round 0: start from scratch

    # ---- 6. Train model ----
    model_name = f"{train_mode}_model_round{round_number}"
    print(f"\nTraining {train_mode.capitalize()} Model: {model_name}")
    start_time = time.time()
    train_cellpose_model(
        train_data=imgs,
        train_labels=masks,
        channels=channels,
        model_save_path=model_save_path,
        n_epochs=n_epochs,
        learning_rate=learning_rate,
        weight_decay=weight_decay,
        model_name=model_name,
        batch_size=batch_size,
        rescale=True,
        normalize=True,
        min_train_masks=min_train_masks,
        momentum=momentum,
        SGD=True,
        compute_flows=True,
        use_GPU=True,
        pretrained_model_path=prev_model_path  # None at round 0, else the previous round's checkpoint
    )
    elapsed = time.time() - start_time
    print(f"Finished {train_mode} model in {str(timedelta(seconds=int(elapsed)))}")


Model save path: training_rounds_results/axial_models/
Training data path: training_data_axial
Resuming axial training from Round 8...

========== ROUND 8 (AXIAL) ==========


Sampling XZ:   0%|          | 0/7 [00:00<?, ?it/s]

Sampling YZ:   0%|          | 0/7 [00:00<?, ?it/s]

Loaded 1400 image/mask pairs from Round 8 (axial)
Training schedule for round 8: 1000 epochs @ LR=0.01
 Fine-tuning from: training_rounds_results/axial_models/models/axial_model_round7

Training Axial Model: axial_model_round8
 Fine-tuning from previous model: training_rounds_results/axial_models/models/axial_model_round7
Training model: axial_model_round8
Training CellPose model using device cuda:0


Training Epochs: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1000/1000 [2:01:17<00:00,  7.28s/it]

 Trained Model saved to: training_rounds_results/axial_models/
 Finished axial model in 2:02:13

========== ROUND 9 (AXIAL) ==========


Sampling XZ:   0%|          | 0/7 [00:00<?, ?it/s]

Sampling YZ:   0%|          | 0/7 [00:00<?, ?it/s]

Loaded 1400 image/mask pairs from Round 9 (axial)
Training schedule for round 9: 1000 epochs @ LR=0.01
 Fine-tuning from: training_rounds_results/axial_models/models/axial_model_round8

Training Axial Model: axial_model_round9
 Fine-tuning from previous model: training_rounds_results/axial_models/models/axial_model_round8
Training model: axial_model_round9
Training CellPose model using device cuda:0


Training Epochs: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1000/1000 [2:02:21<00:00,  7.34s/it]

 Trained Model saved to: training_rounds_results/axial_models/
 Finished axial model in 2:03:13
